# Clase 02: Construyendo memoria y base de conocimiento con RAG

## Módulos instalados

Módulos instalados:
 * `google-genai`: Para conexión con Gemini
 * `python-dotenv`: Para cargar variables de entorno del archivo `.env`
 * `langchain-community`: LangChain es un framework para trabajar con Agentes AI. Este paquete de software, que forma parte del ecosistema de LangChain, está diseñado específicamente para contener integraciones de terceros y herramientas desarrolladas por la comunidad.
 * `pypdf`(Anteriormente PyPDF2) : Es una biblioteca de código abierto escrita en Python, diseñada para manipular, dividir, fusionar, recortar y transformar archivos PDF.
 * `langchain-text-splitters` : Es un paquete de componentes diseñado para dividir documentos extensos en fragmentos más pequeños y manejables (chunks).
 * `langchain-google-genai` : Es un paquete de integración de Python diseñado para conectar el framework LangChain con los modelos de IA generativa de Google, como Gemini.
 * `faiss-cpu` : Es una biblioteca de Meta AI para la búsqueda eficiente de similitud y agrupación de vectores densos en procesadores centrales (CPU), incluso con conjuntos de datos que no caben en la RAM. 

## Creación de RAG

### Agregar API KEY en variables de entorno

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

### Instancia de un cliente de Gemini

In [2]:
from google import genai

cliente = genai.Client()
cliente

In [3]:
respuesta = cliente.models.generate_content(
	model="gemini-2.5-flash",
	contents="Cual es la capital y la ciudad más grande de Turquía?"
)

print(respuesta.text)

La capital de Turquía es **Ankara**.

La ciudad más grande de Turquía es **Estambul**.


### Carga de documentos para el RAG

In [4]:
from langchain_community.document_loaders import PyPDFLoader

In [5]:
documentos = []

for root,_,files in os.walk("data/Clase_01: Assets Inmersión/"):
	for f in files:
		if not f.endswith(".pdf"):
			continue

		ruta = os.path.join(root, f)
		loader = PyPDFLoader(ruta)
		paginas = loader.load()
		documentos.extend(paginas)

In [6]:
documentos[0]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-13T13:30:27+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-13T13:30:27+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data/Clase_01: Assets Inmersión/Carrarurquia_Reporte_Q1_2025.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Carrarurquía\n Reporte trimestral Q1 2025 · Viajes a Turquía (ficticio)\n Periodo: 1 de enero - 31 de marzo de 2025\n Moneda de referencia: dólares estadounidenses (USD)\nEste documento presenta resultados y aprendizajes ficticios del primer trimestre de\n2025 para Carrarurquía, una empresa de viajes especializada en experiencias en\nTurquía. Los datos, cifras y testimonios han sido creados con fines demostrativos y no\nrepresentan operaciones reales.\n Edición: Abril de 2025\n1/15')

In [7]:
len(documentos)

60

### Creación de los fragmentos

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
divisor = RecursiveCharacterTextSplitter(
	chunk_size=400, # Tamaño del fragmento
	chunk_overlap=40, # Superposicion de los fragmentos
	separators=["\n\n", "\n", ". ", " ", ""]
)


fragmentos = divisor.split_documents(documentos)

In [10]:
fragmentos[59]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-13T13:49:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-13T13:49:44+00:00', 'subject': '(unspecified)', 'title': 'Carrarurquía - Reporte Q4 2025', 'trapped': '/False', 'source': 'data/Clase_01: Assets Inmersión/Carrarurquia_Reporte_Q4_2025.pdf', 'total_pages': 15, 'page': 3, 'page_label': '4'}, page_content='Indicador\nResultado\nComentario\nViajeros atendidos\n760\nCrecimiento alineado con cierre de año con mix de ciudad, gastronomía y escapadas.\nIngresos (ventas)\n$2,103,349\nReconocidos al inicio del viaje (criterio simplificado).\nPrecio medio por viajero\n$2,767\nIncluye upsells (globos, hammam, upgrades) cuando aplica.\nMargen bruto\n32.7%\nVariación por tipo de hotel y transporte interno.\nCancelaciones')

### Instancia de embeddings

In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [12]:
# Dependiendo del framework que uses para crear los embeddings, la forma difiere

embeddings = GoogleGenerativeAIEmbeddings(
	model="models/gemini-embedding-001"
)

In [13]:
len(fragmentos)

172

### Creación del Vector Store

In [14]:
# from time import sleep,time

# try:
# 	tiempo_transcurrido = time() - terminado

# 	if 0 < tiempo_transcurrido < 61:
# 		sleep(61 - tiempo_transcurrido)

# except NameError:
# 	pass

# vectorstore1 = FAISS.from_documents(
# 	documents=fragmentos[0:89],
# 	embedding=embeddings
# )

# sleep(61)

# vectorstore2 = FAISS.from_documents(
# 	documents=fragmentos[90:],
# 	embedding=embeddings
# )

# terminado = time()

# TDC Activada

vectorstore1 = FAISS.from_documents(
	documents=fragmentos,
	embedding=embeddings
)

In [15]:
vectorstore1.index.reconstruct(0)

array([-0.00515991,  0.00628808,  0.01395008, ...,  0.01493014,
       -0.00850736, -0.02039895], shape=(3072,), dtype=float32)

### Búsqueda de similitud usando el Vector Store

In [17]:
consulta = "Cual es el paquete de viajes más económico en Carrarurquia?"

resultados = vectorstore1.similarity_search(
	consulta,
	k=3
)

for i in resultados:
	print(i)
	print("\n")

page_content='Carrarurquía
Reporte trimestral Q3 2025 (ficticio)
Periodo: 01/07/2025 - 30/09/2025
Moneda: USD
13
11. Apéndice A: Paquetes y tarifas de referencia
Lista simplificada de precios (USD)
Tarifas ficticias orientativas para paquetes populares. Los importes se muestran por persona en ocupación' metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-13T13:49:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-13T13:49:44+00:00', 'subject': '(unspecified)', 'title': 'Carrarurquía - Reporte Q3 2025', 'trapped': '/False', 'source': 'data/Clase_01: Assets Inmersión/Carrarurquia_Reporte_Q3_2025.pdf', 'total_pages': 15, 'page': 12, 'page_label': '13'}


page_content='Carrarurquía
Reporte trimestral Q2 2025 (ficticio)
Periodo: 01/04/2025 - 30/06/2025
Moneda: USD
13
11. Apéndice A: Paquetes y tarifas de referencia
Lista simplificada de precios (USD)
Tarifas ficticias orientativas para paquetes populare

### Ensamblado del Vector Store y el LLM

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [19]:
# Instancia del LLM

llm = ChatGoogleGenerativeAI(
	model="gemini-2.5-flash",
	temperature=0.2
)


# Instancia de recuperador

retriever = vectorstore1.as_retriever(
	search_kwargs={
		"k" : 4
	}
)

In [20]:
def preguntar_rag(pregunta):
    """Busca contexto relevante en los documentos y genera una respuesta."""
    # Paso 1: Buscar los chunks más relevantes
    docs = retriever.invoke(pregunta)
    contexto = "\n\n---\n\n".join(doc.page_content for doc in docs)

    # Paso 2: Construir el prompt con el contexto encontrado
    prompt = f"""Eres un asistente experto que responde preguntas basándose ÚNICAMENTE
    en el contexto proporcionado. Si la información no está en el contexto,
    di que no tienes suficiente información.

    Contexto: {contexto}

    Pregunta: {pregunta}

    Respuesta:"""

    # Paso 3: Enviar al modelo y devolver la respuesta
    respuesta = llm.invoke(prompt)
    return respuesta.content

### Pruebas de consultas

In [21]:
preguntar_rag("Donde se mantuvo concentrado el mix de productos?")

'El mix de productos se mantuvo concentrado en circuitos combinados (Estambul + Capadocia).'

In [22]:
preguntar_rag("Cuantos mundiales de fútbol tiene brasil?")

'No tengo suficiente información en el contexto para responder a esa pregunta.'

In [23]:
preguntar_rag("Cual es el producto más accesible de Carrarurquia?")

'No tengo suficiente información en el contexto proporcionado para determinar cuál es el producto más accesible de Carrarurquía. El contexto solo indica que el mix de producto se concentra en circuitos combinados (Estambul + Capadocia) y que el precio medio refleja una combinación de hotelería, transporte interno y experiencias.'